# P1 — Stage A ModernBERT + LoRA Training on Kaggle T4

**Before running:**
1. Set accelerator to **GPU T4 x2** in Notebook settings → Accelerator
2. Set Internet access to **ON** (needed to clone repo and download model)
3. Run all cells in order

**Outputs saved to:** `/kaggle/working/models/`

In [ ]:
# Cell 1 — Verify GPU
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Cell 2 — Clone repository
import os
REPO_URL = "https://github.com/Priyrajsinh/P1-Hybrid-Jailbreak-Detector.git"
WORK_DIR = "/kaggle/working/P1-Hybrid-Jailbreak-Detector"

if not os.path.exists(WORK_DIR):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# Cell 3 — Install dependencies
# Use --quiet to suppress verbose output
subprocess.run(
    ["pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
    cwd=WORK_DIR,
)
print("Dependencies installed.")

In [ ]:
# Cell 4 — Build dataset splits (Day 2 data pipeline)
# Skip if data/processed/ already exists from a previous run
import pathlib
if not pathlib.Path("data/processed/train.parquet").exists():
    subprocess.run(
        ["python", "-m", "src.data.pipeline"],
        check=True,
        cwd=WORK_DIR,
    )
    print("Dataset splits built.")
else:
    print("Splits already present, skipping.")

In [ ]:
# Cell 5 — Run Stage A LoRA training
# Logs to MLflow (mlflow.db in working dir) and saves adapter + merged model
subprocess.run(
    [
        "python", "-m", "src.training.train",
        "--config", "config/config.yaml",
    ],
    check=True,
    cwd=WORK_DIR,
)
print("Training complete.")

In [ ]:
# Cell 6 — Export merged model to ONNX
from src.training.export_onnx import export_to_onnx
from src.config import load_config

cfg = load_config("config/config.yaml")
export_to_onnx(
    model_path="models/stage_a_merged",
    output_path="models/stage_a_onnx",
    config=cfg,
)
print("ONNX export complete.")

In [ ]:
# Cell 7 — Save model manifest
import json, pathlib
from src.training.manifest import build_manifest, save_manifest

# Read best_val_f1 from mlflow if available, else use 0.0 as placeholder
try:
    import mlflow
    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment_by_name("p1-hybrid-jailbreak")
    runs = client.search_runs(exp.experiment_id, order_by=["metrics.best_val_f1 DESC"])
    best_f1 = float(runs[0].data.metrics.get("best_val_f1", 0.0)) if runs else 0.0
    n_train = int(runs[0].data.params.get("train_n_train", 0)) if runs else 0
except Exception:
    best_f1, n_train = 0.0, 0

manifest = build_manifest(
    config=cfg,
    training_samples=n_train,
    epochs_completed=int(cfg["training"]["epochs"]),
    best_val_f1=best_f1,
    training_device="T4",
    onnx_exported=pathlib.Path("models/stage_a_onnx").exists(),
    onnx_speedup_ratio=1.0,
)
save_manifest(manifest, "models/manifests/stage_a_manifest.json")
print(json.dumps(manifest, indent=2))

In [ ]:
# Cell 8 — Zip outputs for download
import shutil
shutil.make_archive("/kaggle/working/stage_a_adapter", "zip", "models/stage_a_adapter")
shutil.make_archive("/kaggle/working/stage_a_onnx", "zip", "models/stage_a_onnx")
shutil.copy("models/manifests/stage_a_manifest.json", "/kaggle/working/")

import os
for f in ["/kaggle/working/stage_a_adapter.zip",
          "/kaggle/working/stage_a_onnx.zip",
          "/kaggle/working/stage_a_manifest.json"]:
    size = os.path.getsize(f) / 1024 / 1024
    print(f"{f}: {size:.1f} MB")